In [ ]:
!pip install duckdb pandas

In [ ]:
import duckdb
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

con = duckdb.connect("telecom.duckdb")
con.execute("INSTALL httpfs; LOAD httpfs; SET s3_region='us-west-2';")

path = "s3://ookla-open-data/parquet/performance/type=mobile/year=2025/quarter=1/*.parquet"
print(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}')").df())

        column_name column_type null   key default extra
0           quadkey     VARCHAR  YES  None    None  None
1              tile     VARCHAR  YES  None    None  None
2            tile_x      DOUBLE  YES  None    None  None
3            tile_y      DOUBLE  YES  None    None  None
4        avg_d_kbps      BIGINT  YES  None    None  None
5        avg_u_kbps      BIGINT  YES  None    None  None
6        avg_lat_ms      BIGINT  YES  None    None  None
7   avg_lat_down_ms     INTEGER  YES  None    None  None
8     avg_lat_up_ms     INTEGER  YES  None    None  None
9             tests      BIGINT  YES  None    None  None
10          devices      BIGINT  YES  None    None  None
11          quarter      BIGINT  YES  None    None  None
12             type     VARCHAR  YES  None    None  None
13             year      BIGINT  YES  None    None  None


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/telecom-quality', exist_ok=True)

Mounted at /content/drive


In [ ]:
import os
os.makedirs('/content/drive/MyDrive/telecom-quality', exist_ok=True)

In [ ]:
!pip install duckdb pandas

In [ ]:
import duckdb, pandas as pd

DB = '/content/drive/MyDrive/telecom-quality/telecom.duckdb'
con = duckdb.connect(DB)
con.execute("INSTALL httpfs; LOAD httpfs; SET s3_region='us-west-2';")

# Maharashtra ka approx bounding box
LON_MIN, LON_MAX = 72.6, 80.9
LAT_MIN, LAT_MAX = 15.6, 22.0

con.execute("DROP TABLE IF EXISTS raw_mh")
created = False

for year in range(2022, 2027):
    for q in range(1, 5):
        path = f"s3://ookla-open-data/parquet/performance/type=mobile/year={year}/quarter={q}/*.parquet"
        select = f"""
            SELECT tile_x AS lon, tile_y AS lat, avg_d_kbps, avg_u_kbps,
                   avg_lat_ms, tests, devices, {year} AS year, {q} AS quarter
            FROM read_parquet('{path}')
            WHERE tile_x BETWEEN {LON_MIN} AND {LON_MAX}
              AND tile_y BETWEEN {LAT_MIN} AND {LAT_MAX}
        """
        try:
            if not created:
                con.execute(f"CREATE TABLE raw_mh AS {select}")
                created = True
            else:
                con.execute(f"INSERT INTO raw_mh {select}")
            print(f"{year} Q{q} loaded")
        except Exception as e:
            print(f"{year} Q{q} skipped:", str(e)[:80])

2022 Q1 loaded
2022 Q2 loaded
2022 Q3 loaded
2022 Q4 loaded
2023 Q1 loaded
2023 Q2 loaded
2023 Q3 loaded
2023 Q4 loaded
2024 Q1 loaded
2024 Q2 loaded
2024 Q3 loaded
2024 Q4 loaded
2025 Q1 loaded
2025 Q2 loaded
2025 Q3 loaded


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2025 Q4 loaded
2026 Q1 loaded
2026 Q2 loaded
2026 Q3 loaded
2026 Q4 skipped: IO Error: No files found that match the pattern "s3://ookla-open-data/parquet/pe


In [ ]:
print(con.execute("SELECT year, quarter, COUNT(*) tiles FROM raw_mh GROUP BY 1,2 ORDER BY 1,2").df())
print(con.execute("SELECT MIN(lon), MAX(lon), MIN(lat), MAX(lat) FROM raw_mh").df())

    year  quarter  tiles
0   2022        1  70262
1   2022        2  65301
2   2022        3  64118
3   2022        4  69460
4   2023        1  68544
5   2023        2  69268
6   2023        3  66441
7   2023        4  68274
8   2024        1  62465
9   2024        2  62338
10  2024        3  63098
11  2024        4  62294
12  2025        1  59769
13  2025        2  61276
14  2025        3  60785
15  2025        4  68007
16  2026        1  66587
17  2026        2  66193
18  2026        3  68240
   min(lon)  max(lon)  min(lat)  max(lat)
0   72.6004   80.8951   15.6045   21.9965


In [ ]:
con.execute("DROP TABLE IF EXISTS clean_mh")
con.execute("""
CREATE TABLE clean_mh AS
SELECT
    lon, lat, year, quarter,
    year * 10 + quarter AS yq,
    avg_d_kbps / 1000.0 AS dl_mbps,
    avg_u_kbps / 1000.0 AS ul_mbps,
    avg_lat_ms AS lat_ms,
    tests, devices,
    CASE
        WHEN lat BETWEEN 18.85 AND 19.35 AND lon BETWEEN 72.75 AND 73.15 THEN 'Mumbai Metro'
        WHEN lat BETWEEN 18.40 AND 18.70 AND lon BETWEEN 73.70 AND 74.00 THEN 'Pune'
        WHEN lat BETWEEN 21.05 AND 21.25 AND lon BETWEEN 78.95 AND 79.25 THEN 'Nagpur'
        WHEN lat BETWEEN 19.90 AND 20.10 AND lon BETWEEN 73.70 AND 73.90 THEN 'Nashik'
        ELSE 'Other'
    END AS city
FROM raw_mh
WHERE tests >= 5
  AND devices >= 2
  AND avg_d_kbps BETWEEN 100 AND 1000000
  AND avg_lat_ms BETWEEN 1 AND 2000
""")

print(con.execute("SELECT (SELECT COUNT(*) FROM raw_mh) AS raw_rows, (SELECT COUNT(*) FROM clean_mh) AS clean_rows").df())
print(con.execute("SELECT city, COUNT(*) tiles FROM clean_mh GROUP BY 1 ORDER BY 2 DESC").df())

   raw_rows  clean_rows
0   1242720      370795
           city   tiles
0         Other  283339
1  Mumbai Metro   39869
2          Pune   29464
3        Nagpur   11241
4        Nashik    6882


In [ ]:
q1 = """
SELECT city, year,
       ROUND(SUM(dl_mbps * tests) / SUM(tests), 1) AS avg_dl_mbps,
       ROUND(SUM(lat_ms * tests) / SUM(tests), 1)  AS avg_latency_ms,
       SUM(tests) AS total_tests
FROM clean_mh
WHERE city <> 'Other'
GROUP BY city, year
ORDER BY city, year
"""
print(con.execute(q1).df())

            city  year  avg_dl_mbps  avg_latency_ms  total_tests
0   Mumbai Metro  2022         76.0            33.0     743914.0
1   Mumbai Metro  2023        185.9            28.2     794554.0
2   Mumbai Metro  2024        161.4            28.2     518696.0
3   Mumbai Metro  2025        138.9            27.4     472490.0
4   Mumbai Metro  2026        137.5            26.1     414521.0
5         Nagpur  2022         37.4            50.9      64892.0
6         Nagpur  2023        229.4            41.3      92346.0
7         Nagpur  2024        187.6            35.9      58029.0
8         Nagpur  2025        157.3            37.0      48368.0
9         Nagpur  2026        154.1            37.7      42976.0
10        Nashik  2022         30.6            61.2      39088.0
11        Nashik  2023        198.6            43.6      56473.0
12        Nashik  2024        170.3            48.2      33237.0
13        Nashik  2025        128.6            41.3      33679.0
14        Nashik  2026   

In [ ]:
q2 = """
SELECT city, year, quarter, yq,
       ROUND(SUM(dl_mbps * tests) / SUM(tests), 1) AS avg_dl_mbps,
       ROUND(SUM(lat_ms * tests) / SUM(tests), 1)  AS avg_latency_ms,
       SUM(tests) AS total_tests
FROM clean_mh
WHERE city <> 'Other'
GROUP BY city, year, quarter, yq
ORDER BY city, yq
"""
trend = con.execute(q2).df()
print(trend.head(10))
trend.to_csv('/content/drive/MyDrive/telecom-quality/trend_quarterly.csv', index=False)

           city  year  quarter     yq  avg_dl_mbps  avg_latency_ms  total_tests
0  Mumbai Metro  2022        1  20221         19.9            34.9     152177.0
1  Mumbai Metro  2022        2  20222         21.2            34.5     136682.0
2  Mumbai Metro  2022        3  20223         27.0            36.1     143683.0
3  Mumbai Metro  2022        4  20224        150.1            29.9     311372.0
4  Mumbai Metro  2023        1  20231        202.7            28.5     254726.0
5  Mumbai Metro  2023        2  20232        175.1            28.5     185272.0
6  Mumbai Metro  2023        3  20233        178.0            28.8     182204.0
7  Mumbai Metro  2023        4  20234        181.2            27.0     172352.0
8  Mumbai Metro  2024        1  20241        170.5            27.9     146721.0
9  Mumbai Metro  2024        2  20242        164.5            28.1     129901.0


In [ ]:
q3 = """
WITH t AS (
    SELECT ROUND(lon,5) AS lon, ROUND(lat,5) AS lat, city, year,
           SUM(dl_mbps * tests) / SUM(tests) AS dl
    FROM clean_mh
    WHERE city <> 'Other' AND year BETWEEN 2022 AND 2025
    GROUP BY 1,2,3,4
),
common AS (
    SELECT lon, lat FROM t GROUP BY 1,2 HAVING COUNT(DISTINCT year) = 4
)
SELECT t.city, t.year, COUNT(*) AS tiles, ROUND(AVG(t.dl),1) AS avg_dl_mbps
FROM t JOIN common USING (lon, lat)
GROUP BY 1,2 ORDER BY 1,2
"""
print(con.execute(q3).df())

            city  year  tiles  avg_dl_mbps
0   Mumbai Metro  2022   2107         64.3
1   Mumbai Metro  2023   2107        181.0
2   Mumbai Metro  2024   2107        166.9
3   Mumbai Metro  2025   2107        144.8
4         Nagpur  2022    597         33.3
5         Nagpur  2023    597        215.3
6         Nagpur  2024    597        185.4
7         Nagpur  2025    597        155.3
8         Nashik  2022    366         28.8
9         Nashik  2023    366        188.8
10        Nashik  2024    366        166.0
11        Nashik  2025    366        128.3
12          Pune  2022   1559         64.2
13          Pune  2023   1559        177.0
14          Pune  2024   1559        165.1
15          Pune  2025   1559        136.1


In [ ]:
q4 = """
SELECT ROUND(lon,4) AS lon, ROUND(lat,4) AS lat,
       ROUND(SUM(dl_mbps * tests) / SUM(tests), 1) AS avg_dl_mbps,
       ROUND(SUM(lat_ms * tests) / SUM(tests), 1)  AS avg_latency_ms,
       SUM(tests) AS total_tests
FROM clean_mh
WHERE city = 'Pune' AND year = 2026
GROUP BY 1,2
HAVING SUM(tests) >= 30
ORDER BY avg_dl_mbps ASC
LIMIT 15
"""
print(con.execute(q4).df())

        lon      lat  avg_dl_mbps  avg_latency_ms  total_tests
0   73.8254  18.4353         20.0            88.6         40.0
1   73.8199  18.6541         21.3           139.6        173.0
2   73.7210  18.5551         22.6            30.2         81.0
3   73.8364  18.6957         23.1            35.6         39.0
4   73.7265  18.5760         24.9            36.4         35.0
5   73.7210  18.6176         27.0            32.1         36.0
6   73.7430  18.5447         27.8            44.7         40.0
7   73.9792  18.5135         29.2            35.0         34.0
8   73.7265  18.5708         30.7            35.5         84.0
9   73.7320  18.5760         30.8            34.8         30.0
10  73.7210  18.6697         31.0            51.2         49.0
11  73.9462  18.6072         35.4            24.0         37.0
12  73.8199  18.6488         36.9            56.6         82.0
13  73.9957  18.5499         37.1            68.7         58.0
14  73.7100  18.6280         39.6            42.0      

In [ ]:
base = '/content/drive/MyDrive/telecom-quality/'

con.execute(q1).df().to_csv(base + 'city_year.csv', index=False)

con.execute("""
SELECT city, ROUND(lon,4) AS lon, ROUND(lat,4) AS lat,
       ROUND(SUM(dl_mbps * tests) / SUM(tests), 1) AS avg_dl_mbps,
       ROUND(SUM(lat_ms * tests) / SUM(tests), 1)  AS avg_latency_ms,
       SUM(tests) AS total_tests
FROM clean_mh
WHERE year = 2026
GROUP BY 1,2,3
HAVING SUM(tests) >= 10
""").df().to_csv(base + 'tiles_2026.csv', index=False)
print("saved")

saved


In [ ]:
q5 = """
SELECT CASE WHEN city = 'Other' THEN 'Rest of region' ELSE 'Major cities' END AS area,
       year,
       ROUND(SUM(dl_mbps * tests) / SUM(tests), 1) AS avg_dl_mbps,
       ROUND(SUM(lat_ms * tests) / SUM(tests), 1)  AS avg_latency_ms,
       COUNT(*) AS tiles
FROM clean_mh
GROUP BY 1,2 ORDER BY 1,2
"""
print(con.execute(q5).df())

             area  year  avg_dl_mbps  avg_latency_ms  tiles
0    Major cities  2022         70.6            37.3  18301
1    Major cities  2023        189.1            32.1  19218
2    Major cities  2024        164.8            32.6  17960
3    Major cities  2025        138.7            30.6  17741
4    Major cities  2026        135.6            29.3  14236
5  Rest of region  2022         43.9            48.3  62663
6  Rest of region  2023        183.2            40.4  66864
7  Rest of region  2024        171.6            43.0  56965
8  Rest of region  2025        142.0            41.5  53240
9  Rest of region  2026        136.8            39.4  43607


In [ ]:
con.execute(q5).df().to_csv('/content/drive/MyDrive/telecom-quality/area_gap.csv', index=False)

In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/telecom-quality'))

['telecom.duckdb', 'trend_quarterly.csv', 'city_year.csv', 'tiles_2026.csv', 'area_gap.csv']
